In [ ]:
import json
import pandas as pd
from google import genai
from google.genai import types

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

# Confirm which Flash models are active on your key
for m in client.models.list():
    if "flash" in m.name.lower():
        print(m.name)

In [9]:
MODEL_NAME = "gemini-3.5-flash"
api_call_count = 0

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

SYSTEM_TEXT = (
    "You are a healthcare operations analyst for CareLoop Clinics in India. "
    "Classify post-appointment call-centre notes accurately into structured JSON without guessing."
)

def build_craft_prompt(note_text: str) -> str:
    return f"""
Context: CareLoop Clinics' call-centre team phones patients who missed outpatient appointments and records short follow-up notes (sometimes in English or Hinglish).
Role: Act as a clinical operations analyst extracting structured no-show reasons for front-desk follow-up.
Action: Read the patient follow-up note below and classify it into exactly ONE root_cause from this allowed list:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

If the note is ambiguous, unclear, or does not clearly fit the first five categories, output "other" rather than guessing. Also provide a confidence score between 0 and 1, a concise summary of at most 25 words, and a practical recommended_action for the front desk.
Tone: Objective, concise, and operational.
Format: Return valid JSON matching the required schema with keys root_cause, confidence, summary, and recommended_action.

Call-centre note: "{note_text}"
""".strip()

def analyse_note(note_text: str, custom_prompt: str = None) -> tuple[dict, str]:
    """Calls Gemini Flash and returns (parsed_dict, raw_response_text)."""
    global api_call_count
    prompt_to_send = custom_prompt if custom_prompt is not None else build_craft_prompt(note_text)

    api_call_count += 1
    print(f"Calling {MODEL_NAME} (call #{api_call_count})...", flush=True)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_to_send,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=2000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="low")
        )
    )
    raw_text = response.text
    parsed = json.loads(raw_text)
    return parsed, raw_text

# Load notes and run Task 4.1 on N001
notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].values[0]

n001_result, _ = analyse_note(n001_text)
print("Note N001 text:", n001_text)
print("Parsed output:", json.dumps(n001_result, indent=2))
print("Total API calls so far:", api_call_count)

Calling gemini-3.5-flash (call #1)...
Note N001 text: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
Parsed output: {
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient forgot appointment due to not receiving a reminder message and requested a rebooking for next week.",
  "recommended_action": "Reschedule patient's appointment for next week and verify SMS/WhatsApp reminder contact preferences in the system."
}
Total API calls so far: 1


In [10]:
import time
from pydantic import BaseModel, Field, ValidationError

# (a) Define Pydantic validation model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# (b) & (c) Loop over the first 10 rows with blank check, validation, and rate-limit pause
results = []

for idx, row in notes_df.head(10).iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    # Check for missing or blank notes before calling the API
    if pd.isna(note_text) or str(note_text).strip() == "":
        print(f"{note_id}: Blank note -> skipped_blank (no API call)")
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    # Non-blank note: call API and validate without repairing invalid responses
    raw_resp = ""
    parsed_data = {}
    status = "invalid"

    try:
        parsed_data, raw_resp = analyse_note(str(note_text))

        # Validate structure/types with Pydantic and check allowed root_cause list
        insight = FollowupInsight(**parsed_data)
        if insight.root_cause in ALLOWED_CAUSES:
            status = "valid"
        else:
            status = "invalid"
            print(f"{note_id}: Invalid root_cause '{insight.root_cause}'")

    except (ValidationError, json.JSONDecodeError, Exception) as e:
        status = "invalid"
        print(f"{note_id}: Marked invalid ({type(e).__name__}: {e})")

    # Keep whatever fields were returned; leave missing fields empty
    results.append({
        "note_id": note_id,
        "appointment_id": appt_id,
        "root_cause": parsed_data.get("root_cause", ""),
        "confidence": parsed_data.get("confidence", ""),
        "summary": parsed_data.get("summary", ""),
        "recommended_action": parsed_data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_resp
    })

    print(f"{note_id}: {status} -> {parsed_data.get('root_cause', 'N/A')}")

    # Respect free-tier rate limit (5 requests/min)
    time.sleep(13)

# (d) Save notes_analysed.csv
analysed_df = pd.DataFrame(results)
analysed_df.to_csv("notes_analysed.csv", index=False)

print(f"\nSaved notes_analysed.csv with shape: {analysed_df.shape}")
print(f"Total API calls across 4.1 & 4.2: {api_call_count}")
print("\n--- Summary Table ---")
print(analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))

Calling gemini-3.5-flash (call #2)...
N001: valid -> forgot_or_no_reminder
Calling gemini-3.5-flash (call #3)...
N002: valid -> distance_or_transport
Calling gemini-3.5-flash (call #4)...
N003: valid -> cost_or_fee
N004: Blank note -> skipped_blank (no API call)
Calling gemini-3.5-flash (call #5)...
N005: valid -> illness_or_emergency
Calling gemini-3.5-flash (call #6)...
N006: valid -> work_or_schedule
Calling gemini-3.5-flash (call #7)...
N007: valid -> forgot_or_no_reminder
Calling gemini-3.5-flash (call #8)...
N008: valid -> distance_or_transport
Calling gemini-3.5-flash (call #9)...
N009: valid -> other
Calling gemini-3.5-flash (call #10)...
N010: valid -> work_or_schedule

Saved notes_analysed.csv with shape: (10, 8)
Total API calls across 4.1 & 4.2: 10

--- Summary Table ---
note_id appointment_id            root_cause  confidence validation_status
   N001          A0872 forgot_or_no_reminder        0.98             valid
   N002          A0897 distance_or_transport        0.98 

In [11]:
# (a) Feed two deliberately invalid replies to Pydantic and catch ValidationError
bad_confidence_payload = {
    "root_cause": "cost_or_fee",
    "confidence": 1.7,
    "summary": "Patient mentioned the consultation fee was unaffordable.",
    "recommended_action": "Explain available clinic fee options."
}

bad_missing_field_payload = {
    "root_cause": "work_or_schedule",
    "confidence": 0.90,
    "summary": "Patient could not leave office meeting in time."
    # recommended_action intentionally omitted
}

print("--- (a) Deliberate Pydantic ValidationError Tests ---")
for test_name, payload in [
    ("Confidence = 1.7 (exceeds upper bound)", bad_confidence_payload),
    ("Missing recommended_action field", bad_missing_field_payload)
]:
    try:
        FollowupInsight(**payload)
    except ValidationError as e:
        print(f"\nCaught expected error for [{test_name}]:")
        print(e)

# (b) Compare V1 vs. V2 prompts on N007 (Hinglish) and N009 (ambiguous) at temperature=0
print("\n--- (b) Prompt Comparison: V1 vs. V2 on N007 & N009 ---")
comparison_rows = []

for target_id in ["N007", "N009"]:
    note_txt = notes_df.loc[notes_df["note_id"] == target_id, "note_text"].values[0]

    # V1: Minimal prompt without allowed categories or anti-guessing rule
    v1_prompt = f"Classify this note:\n{note_txt}"
    v1_res, _ = analyse_note(note_txt, custom_prompt=v1_prompt)
    v1_cause = v1_res.get("root_cause", "")
    comparison_rows.append({
        "note_id": target_id,
        "prompt": "V1 (Basic)",
        "root_cause": v1_cause,
        "in_allowed_list": v1_cause in ALLOWED_CAUSES,
        "confidence": v1_res.get("confidence", None)
    })
    time.sleep(13)

    # V2: Full CRAFT prompt with allowed list and "other" fallback rule
    v2_prompt = build_craft_prompt(note_txt)
    v2_res, _ = analyse_note(note_txt, custom_prompt=v2_prompt)
    v2_cause = v2_res.get("root_cause", "")
    comparison_rows.append({
        "note_id": target_id,
        "prompt": "V2 (CRAFT)",
        "root_cause": v2_cause,
        "in_allowed_list": v2_cause in ALLOWED_CAUSES,
        "confidence": v2_res.get("confidence", None)
    })
    time.sleep(13)

prompt_comp_df = pd.DataFrame(comparison_rows)
print("\n" + prompt_comp_df.to_string(index=False))

# (c) Print total API calls across Tasks 4.1–4.3
print(f"\n--- (c) Total API Calls Used (Tasks 4.1–4.3): {api_call_count} ---")

--- (a) Deliberate Pydantic ValidationError Tests ---

Caught expected error for [Confidence = 1.7 (exceeds upper bound)]:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Caught expected error for [Missing recommended_action field]:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...ffice meeting in time.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

--- (b) Prompt Comparison: V1 vs. V2 on N007 & N009 ---
Calling gemini-3.5-flash (call #11)...
Calling gemini-3.5-flash (call #12)...
Calling gemini-3.5-flash (call #13)...
Calling gemini-3.5-flash (call #14)...

note_id     prompt                               root_cause  in_allowed_list  confidence
   N007 V1 (Basic)    Pati

**Task 4.3 Prompt Deployment Decision:**
I would deploy **V2 (the CRAFT prompt)** because JSON mode alone only enforces field names and data types, whereas V2 constrains `root_cause` to the exact six allowed operational categories, handles Hinglish (`N007` → `forgot_or_no_reminder`), and prevents hallucinated guesses on ambiguous notes (`N009` → `other`).

In [12]:
# Merge analysed notes with cleaned appointment records
appts_clean = pd.read_csv("appointments_clean.csv")
analysed_notes = pd.read_csv("notes_analysed.csv")

merged_df = analysed_notes.merge(appts_clean, on="appointment_id", how="inner")
print(f"Merged dataset shape: {merged_df.shape}")

# Cross-tabulate root_cause against reminder_sent
ct = pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"], margins=True)
print("\n--- Cross-Tabulation: root_cause vs. reminder_sent ---")
print(ct)

Merged dataset shape: (10, 23)

--- Cross-Tabulation: root_cause vs. reminder_sent ---
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10
